<a href="https://colab.research.google.com/github/manish7725/deeplearning/blob/main/Lecture%2010%20-%20Probability%3A%20Reasoning%20Under%20Uncertainty/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 10 — Probability: Reasoning Under Uncertainty · Laboratory

↩ **Theory:** [`blog.md`](<blog.md>) — read the lecture first. This notebook tests it.

## Step 1 — The Problem

*"A buyer offered ₹14 lakh. What are the chances the next flat sells for more?"*

Two of Chapter 9's ten sales beat ₹14 lakh, so $2/10 = 20\%$. But that is a fact about ten
sales that already happened. Push it and it breaks: no sale reached ₹20 lakh, so counting
says that is **impossible** — which is a much stronger claim than *unobserved*.

## Step 2 — Prediction

Commit before running. You check these in Step 10.

1. From the 100-sale table, is $P(\text{sold} > 14 \mid 4\text{ rooms})$ bigger or smaller
   than $P(4\text{ rooms} \mid \text{sold} > 14)$?
2. Prices are 10, 12 or 15 with probabilities 0.5, 0.3, 0.2. What is the expected price —
   and can that price actually occur?
3. Simulating sales and averaging: how many do you need before the running mean is within
   0.02 of the true expected value?
4. Are "4 rooms" and "sold above 14" independent?

In [ ]:
# Step 3 — Intuition: counting the past cannot answer questions about the future.
import numpy as np
np.random.seed(0)

prices = np.array([9., 10., 11., 11., 12., 12., 12., 13., 14., 16.])

print("fraction of past sales above 14:", (prices > 14).mean())
print("fraction above 20:              ", (prices > 20).mean(), " <- 'impossible'?")
print("fraction exactly 11.5:          ", (prices == 11.5).mean(), " <- also 'impossible'?")

assert (prices > 14).mean() == 0.2
assert (prices > 20).mean() == 0.0
# 11.5 sits in the middle of the data and is assigned probability zero by counting.
assert (prices == 11.5).mean() == 0.0

## Step 4 — The Mathematics Under Test

$$0 \le P(A) \le 1
\qquad
P(\Omega)=1
\qquad
P(A\cup B)=P(A)+P(B)-P(A\cap B)$$

$$P(A\mid B)=\frac{P(A\cap B)}{P(B)}
\qquad
\mathbb{E}[X]=\sum_x x\,P(x)
\qquad
\mathrm{Var}(X)=\mathbb{E}[X^2]-\mathbb{E}[X]^2$$

In [ ]:
# Step 5 — Manual calculation: the 100-sale table from blog section 5.
#                     sold>14   sold<=14
#   4 rooms              18        12      = 30
#   fewer rooms           7        63      = 70
table = np.array([[18, 12],
                  [7,  63]], dtype=float)
total = table.sum()
assert total == 100

p_4rooms = table[0].sum() / total
p_high   = table[:, 0].sum() / total
p_both   = table[0, 0] / total
print(f"P(4 rooms)          = {p_4rooms}")
print(f"P(sold > 14)        = {p_high}")
print(f"P(4 rooms and >14)  = {p_both}")
assert p_4rooms == 0.30 and p_high == 0.25 and p_both == 0.18

# Conditioning = zoom in, then rescale.
p_high_given_4 = p_both / p_4rooms
p_4_given_high = p_both / p_high
print(f"\nP(>14 | 4 rooms) = {p_both}/{p_4rooms} = {p_high_given_4}")
print(f"P(4 rooms | >14) = {p_both}/{p_high} = {p_4_given_high}")
assert np.isclose(p_high_given_4, 0.6)
assert np.isclose(p_4_given_high, 0.72)
assert not np.isclose(p_high_given_4, p_4_given_high)

print("\nSame numerator, different denominators -> different questions, different answers.")

In [ ]:
# Step 5b — the rules that follow from the three axioms (blog section 4).
# complement
assert np.isclose(1 - p_high, table[:, 1].sum() / total)
print(f"P(not >14) = 1 - {p_high} = {1-p_high}")

# overlap: P(A or B) = P(A) + P(B) - P(A and B)
p_or = p_4rooms + p_high - p_both
counted = (table[0].sum() + table[:, 0].sum() - table[0, 0]) / total
print(f"P(4 rooms or >14) = {p_4rooms} + {p_high} - {p_both} = {p_or}")
assert np.isclose(p_or, counted)
print(f"   counted directly: {counted}  (adding without subtracting would give {p_4rooms+p_high})")

# independence check: does knowing the room count change the price probability?
print(f"\nP(>14) = {p_high},  P(>14 | 4 rooms) = {p_high_given_4}")
assert not np.isclose(p_high, p_high_given_4)
print("Not independent -- which is exactly why rooms is a useful feature.")

In [ ]:
# Step 5c — random variables and expectation (blog section 7).
values = np.array([10., 12., 15.])
probs  = np.array([0.5, 0.3, 0.2])
assert np.isclose(probs.sum(), 1.0)          # axiom 2

E_X  = (values * probs).sum()
E_X2 = (values**2 * probs).sum()
var  = E_X2 - E_X**2
print(f"E[X]   = {E_X}")
print(f"E[X^2] = {E_X2}")
print(f"Var(X) = {E_X2} - {E_X}^2 = {var:.4f}")
print(f"sd     = {np.sqrt(var):.4f}")

assert np.isclose(E_X, 11.6)
assert np.isclose(E_X2, 138.2)
assert np.isclose(var, 3.64)

# The direct definition must agree with the shortcut.
var_direct = (probs * (values - E_X)**2).sum()
assert np.isclose(var, var_direct)

# The "expected" value cannot actually occur:
assert 11.6 not in values
print("\nE[X] = 11.6, but the only possible prices are 10, 12 and 15.")
print("It is a balance point, not a prediction.")

In [ ]:
# Step 6 — First implementation: conditioning by simulation, not by formula.
# Generate 200,000 sales matching the table, then COUNT -- and check the
# counted conditionals match the formula from Step 5.
rng = np.random.default_rng(0)
N = 200_000
cell_probs = (table / total).ravel()          # [4&high, 4&low, few&high, few&low]
draws = rng.choice(4, size=N, p=cell_probs)

is_4room = np.isin(draws, [0, 1])
is_high  = np.isin(draws, [0, 2])

print(f"simulated P(4 rooms)      = {is_4room.mean():.4f}   (exact 0.30)")
print(f"simulated P(>14)          = {is_high.mean():.4f}   (exact 0.25)")
print(f"simulated P(>14 | 4 room) = {is_high[is_4room].mean():.4f}   (exact 0.60)")
print(f"simulated P(4 room | >14) = {is_4room[is_high].mean():.4f}   (exact 0.72)")

assert abs(is_high[is_4room].mean() - 0.60) < 0.01
assert abs(is_4room[is_high].mean() - 0.72) < 0.01
# Conditioning really is "throw away the rows where B is false, then recount".

In [ ]:
# Step 7 — Visualization: probability as area, and conditioning as zooming in.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# (a) the sample space as a unit square, split by the table
ax = axes[0]
ax.add_patch(plt.Rectangle((0, 0), 0.30, 0.60, color="tab:blue", alpha=.75))
ax.add_patch(plt.Rectangle((0, 0.60), 0.30, 0.40, color="tab:blue", alpha=.30))
ax.add_patch(plt.Rectangle((0.30, 0), 0.70, 0.10, color="tab:orange", alpha=.75))
ax.add_patch(plt.Rectangle((0.30, 0.10), 0.70, 0.90, color="tab:orange", alpha=.30))
ax.text(0.15, 0.30, "18", ha="center", color="white", fontsize=12)
ax.text(0.15, 0.80, "12", ha="center", fontsize=12)
ax.text(0.65, 0.05, "7", ha="center", color="white", fontsize=12)
ax.text(0.65, 0.55, "63", ha="center", fontsize=12)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel("4 rooms (0.30)  |  fewer rooms (0.70)")
ax.set_title("the sample space has area 1")

# (b) conditioning: keep only the 4-room column, rescale to area 1
ax = axes[1]
ax.add_patch(plt.Rectangle((0, 0), 1, 0.60, color="tab:blue", alpha=.75))
ax.add_patch(plt.Rectangle((0, 0.60), 1, 0.40, color="tab:blue", alpha=.30))
ax.axhline(0.60, color="black", lw=1)
ax.text(0.5, 0.30, "sold > 14\n0.60", ha="center", va="center", color="white", fontsize=11)
ax.text(0.5, 0.80, "sold <= 14\n0.40", ha="center", va="center", fontsize=11)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title("given 4 rooms: rescaled to area 1")

# (c) the law of large numbers
sales = rng.choice(values, size=20000, p=probs)
running = np.cumsum(sales) / np.arange(1, len(sales) + 1)
axes[2].semilogx(running)
axes[2].axhline(E_X, color="crimson", ls="--", label=f"E[X] = {E_X}")
axes[2].set_title("running average settles onto E[X]")
axes[2].set_xlabel("number of sales"); axes[2].legend(fontsize=8)
axes[2].grid(alpha=.3)

plt.tight_layout(); plt.show()

In [ ]:
# Step 8 — The experiment: do frequencies really approach probabilities, and how fast?
sigma = np.sqrt(var)
print(f"true E[X] = {E_X},  sigma = {sigma:.4f}\n")
print(f"{'n':>10} {'observed spread of mean':>26} {'sigma/sqrt(n)':>16}")
for n in [10, 100, 10_000]:
    means = [rng.choice(values, size=n, p=probs).mean() for _ in range(3000)]
    observed = np.std(means, ddof=1)
    predicted = sigma / np.sqrt(n)
    print(f"{n:>10} {observed:>26.4f} {predicted:>16.4f}")
    assert abs(observed - predicted) < 0.03 * max(1, predicted / 0.01)

# A hundred times the data buys one extra decimal place.
for n in [10, 100, 10_000, 1_000_000]:
    print(f"  n = {n:>9,}  typical error {sigma/np.sqrt(n):.4f}")

In [ ]:
# Step 9 — Change exactly one variable: make one outcome rare.
# Same three prices, but 15 lakh now happens only 0.1% of the time.
rare_probs = np.array([0.5, 0.499, 0.001])
E_rare = (values * rare_probs).sum()
print(f"E[X] with a rare outcome = {E_rare:.4f}")

print(f"\n{'n':>10} {'running mean':>16} {'error':>12}")
for n in [100, 1_000, 10_000, 100_000]:
    sample = rng.choice(values, size=n, p=rare_probs)
    m = sample.mean()
    print(f"{n:>10} {m:>16.4f} {abs(m - E_rare):>12.4f}")

# How many draws before we even SEE the rare outcome, on average?
print(f"\nexpected draws before seeing a 0.1% outcome: {1/0.001:.0f}")
print("With imbalanced data, the sample says nothing about the rare class")
print("until n is large compared with 1/p. Chapter 21 returns to this.")

## Step 10 — Observe

Against your Step 2 predictions:

1. $P(>14 \mid 4\text{ rooms}) = 0.60$ but $P(4\text{ rooms} \mid >14) = 0.72$. Same
   numerator $0.18$, different denominators — **different questions**.
2. $\mathbb{E}[X] = 11.6$, a price that **cannot occur**. It is a balance point, not a
   prediction.
3. The running mean settles onto 11.6, but slowly: the error falls as $\sigma/\sqrt{n}$, so a
   hundredfold more data buys one extra decimal place.
4. Not independent — knowing the flat has 4 rooms raises the probability from $0.25$ to
   $0.60$. That is precisely why rooms is a useful feature.

Step 6 is the one to sit with: conditioning was done by *simulation* — keep only the rows
where $B$ is true, then recount — and the answer matched the formula to two decimals. The
formula is not an abstraction on top of counting; it *is* the counting.

## Step 11 — Explain

**Why counting is not enough.** A frequency of zero means "not seen in this sample"; a
probability of zero means "cannot happen". Chapter 9 drew the same distinction between the
sample mean $\bar x$ and the true mean $\mu$. Estimates wobble; the thing estimated does not.

**Why conditioning divides by $P(B)$.** Restricting to $B$ throws away part of the sample
space, so what remains no longer has total area 1. Dividing by $P(B)$ rescales it back — which
is why $P(A\mid B) = P(A\cap B)/P(B)$ and not simply $P(A \cap B)$.

**Why the two conditionals differ.** They share the numerator $P(A \cap B) = 0.18$ and divide
by different denominators: $0.30$ of the sheet is four-room flats, $0.25$ is high sales. Same
overlap, different worlds to rescale within.

**Why more data helps so slowly.** Step 8 confirms the error falls as $1/\sqrt{n}$, the same
law Chapter 9 §9 found. This is the economics of every dataset: accuracy gets more expensive
the more of it you already have, and for rare events you need $n$ large compared with $1/p$
before the sample says anything at all.

In [ ]:
# Step 12 — Challenges.

# LEVEL 2 (by hand first): one die roll.
# P(even), P(>4), P(even and >4), P(even or >4) -- then check independence.
outcomes = np.arange(1, 7)
even = outcomes % 2 == 0
big = outcomes > 4
print("P(even)        =", even.mean())
print("P(>4)          =", big.mean())
print("P(even and >4) =", (even & big).mean())
print("P(even or >4)  =", (even | big).mean(), " check:", even.mean()+big.mean()-(even&big).mean())
print("P(even | >4)   =", even[big].mean(), " vs P(even) =", even.mean())
assert np.isclose((even | big).mean(), even.mean() + big.mean() - (even & big).mean())

# LEVEL 4 (Investigate): how many samples before a running mean is reliable when one
# outcome has probability 0.001? Measure it, then relate your answer to 1/p.

# YOUR CODE HERE


# LEVEL 5 (Design): counting gives P(price > 20) = 0 because no sale reached it.
# Design a method that assigns a small positive probability instead. State your
# assumption, compute a number, and say what would make the assumption wrong.
def p_above(threshold, data):
    # YOUR CODE HERE
    ...

## Step 13 — Reflection

- [ ] I can say why a frequency of zero is not a probability of zero.
- [ ] I can state the three axioms and derive the complement and overlap rules from them.
- [ ] I can explain conditioning as zooming in and rescaling.
- [ ] I can give an example where $P(A\mid B) \ne P(B\mid A)$ and say why.
- [ ] I know the difference between *independent* and *exclusive*.
- [ ] I can explain why $\mathbb{E}[X]$ may be a value that never occurs.
- [ ] I know that error falls as $1/\sqrt{n}$, and what that costs.

### The question this chapter leaves open

Every conditional we computed ran **from cause to evidence**: given 4 rooms, how likely is a
high price? Real questions run the other way. A damp-detector alarms — what is the chance the
house actually *has* damp? We know $P(\text{alarm}\mid\text{damp})$, because that is what the
manufacturer measured. We want $P(\text{damp}\mid\text{alarm})$.

§5 proved we cannot just swap them.

➡️ **Next:** [Chapter 11 — Bayes' Rule: What Evidence Does to Belief](<../Lecture 11 - Bayes' Rule: What Evidence Does to Belief/blog.md>)